In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as func
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
import sys
import time

def computeCosineSimilarity(spark, data):
    pairScores = data \
      .withColumn("xx", func.col("rating1") * func.col("rating1")) \
      .withColumn("yy", func.col("rating2") * func.col("rating2")) \
      .withColumn("xy", func.col("rating1") * func.col("rating2"))

    calculateSimilarity = pairScores \
      .groupBy("movie1", "movie2") \
      .agg( \
        func.sum(func.col("xy")).alias("numerator"), \
        (func.sqrt(func.sum(func.col("xx"))) * func.sqrt(func.sum(func.col("yy")))).alias("denominator"), \
        func.count(func.col("xy")).alias("numPairs")
      )

    result = calculateSimilarity \
      .withColumn("score", \
        func.when(func.col("denominator") != 0, func.col("numerator") / func.col("denominator")) \
          .otherwise(0) \
      ).select("movie1", "movie2", "score", "numPairs")

    return result

def getMovieName(movieNames, movieId):
    result = movieNames.filter(func.col("movieID") == movieId) \
        .select("movieTitle").collect()[0]
    return result[0]

# Criar sessão Spark
print("Iniciando Spark Session...")
spark = SparkSession.builder.remote("sc://localhost:15002").appName("MovieSimilarities").getOrCreate()


print("SPARK INICIADO COM SUCESSO")
print(f"Application Name: {spark.sparkContext.appName}")
print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print("\nACESSE A INTERFACE WEB:")
print("   http://localhost:4040")
print("   http://localhost:15002")

# Schemas
movieNamesSchema = StructType([
    StructField("movieID", IntegerType(), True),
    StructField("movieTitle", StringType(), True)
])

moviesSchema = StructType([
    StructField("userID", IntegerType(), True),
    StructField("movieID", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", LongType(), True)
])

# Carregar dados
print("Carregando dados...")
movieNames = spark.read \
    .option("sep", "|") \
    .option("charset", "ISO-8859-1") \
    .schema(movieNamesSchema) \
    .csv("../Dados/ml-100k/u.item")

movies = spark.read \
    .option("sep", "\t") \
    .schema(moviesSchema) \
    .csv("../Dados/ml-100k/u.data")

print(f"Filmes carregados: {movieNames.count()}")
print(f"Avaliações carregadas: {movies.count()}")

# Processar dados
print("\nProcessando similaridades...")
ratings = movies.select("userId", "movieId", "rating")

moviePairs = ratings.alias("ratings1") \
    .join(ratings.alias("ratings2"),
          (func.col("ratings1.userId") == func.col("ratings2.userId")) \
          & (func.col("ratings1.movieId") < func.col("ratings2.movieId"))) \
    .select(func.col("ratings1.movieId").alias("movie1"),
            func.col("ratings2.movieId").alias("movie2"),
            func.col("ratings1.rating").alias("rating1"),
            func.col("ratings2.rating").alias("rating2"))

moviePairSimilarities = computeCosineSimilarity(spark, moviePairs).cache()

print("Similaridades calculadas")
print("\nAmostra dos resultados:")
moviePairSimilarities.orderBy(func.col("score").desc()).show(10)

# Buscar filmes similares se ID fornecido
if len(sys.argv) > 1:
    scoreThreshold = 0.97
    coOccurrenceThreshold = 50.0
    movieID = int(sys.argv[1])

    filteredResults = moviePairSimilarities.filter(
        ((func.col("movie1") == movieID) | (func.col("movie2") == movieID)) &
        (func.col("score") > scoreThreshold) &
        (func.col("numPairs") > coOccurrenceThreshold))

    results = filteredResults.sort(func.col("score").desc()).take(10)

    print(f"\nTop 10 filmes similares a: {getMovieName(movieNames, movieID)}")
    print("="*60)

    for result in results:
        similarMovieID = result.movie1 if result.movie1 != movieID else result.movie2
        print(f"{getMovieName(movieNames, similarMovieID)}")
        print(f"   Score: {result.score:.4f} | Força: {result.numPairs}")

# Manter aplicação rodando
print("INTERFACE WEB ATIVA")
print("Acesse: http://localhost:4040")
print("\nA aplicação continuará rodando...")

try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print("\nEncerrando aplicação Spark...")
    spark.stop()
    print("Encerrado com sucesso")

In [3]:
!pip install "grpcio>=1.48.1" --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 17.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.8.0
    Uninstalling typing_extensions-4.8.0:
      Successfully uninstalled typing_extensions-4.8.0


In [2]:
conda install -c conda-forge grpcio grpcio-status googleapis-common-protos


Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1006)'))': /conda-forge/linux-64/current_repodata.json

| Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1006)'))': /conda-forge/linux-64/current_repodata.json

Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1006)'))': /conda-forge/noarch/current_repodata.json

| Retrying (Retry(total=0, connect=None, read=None, redir

In [3]:
pip install pyspark[connect,sql,pandas_on_spark,ml,mllib] --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 4.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 5.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 6.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.24.3
    Uninstalling protobuf-4.24.3:
      Successfully uninstalled protobuf-4.24.3
Note: you may need to restart the kernel to use updated packages.
